# ⚔️ MOTUS-VIGILUS — Frégate U-ALPHA : L'Auspex
> Extraction cinétique : Vidéo .mp4 → Données de mouvement .npz

**Pipeline** : 📹 .mp4 → [MediaPipe Pose 3D] → [Quaternions] → [Smoothing] → 📦 .npz

**Stockage** : Tout est sur Google Drive (`MOTUS-VIGILUS/U-ALPHA/`)

**Instructions** :
1. Exécuter l'installation (monte Drive + installe les dépendances)
2. Déposer votre vidéo .mp4 dans `Drive/MOTUS-VIGILUS/U-ALPHA/inputs/`
3. Configurer les paramètres (FPS, Lissage, Root Motion)
4. Sélectionner la vidéo à traiter
5. Lancer l'extraction
6. Récupérer les .npz dans `Drive/MOTUS-VIGILUS/U-ALPHA/outputs/`

In [ ]:
# ═══════ INSTALLATION + MONTAGE DRIVE — FRÉGATE U-ALPHA ═══════
from google.colab import drive
drive.mount('/content/drive')

# Structure Drive
import os
DRIVE_BASE = "/content/drive/My Drive/MOTUS-VIGILUS/U-ALPHA"
INPUTS_DIR = f"{DRIVE_BASE}/inputs"
OUTPUTS_DIR = f"{DRIVE_BASE}/outputs"
MODELS_DIR = f"{DRIVE_BASE}/models"

for d in [INPUTS_DIR, OUTPUTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# Installer les dépendances
!pip install -q mediapipe numpy scipy scenedetect[opencv] opencv-python-headless

# Cloner le codebase (1 fois, ou git pull si déjà cloné)
REPO_DIR = "/content/MOTUS-VIGILUS"
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone -q https://github.com/kioka8877-ux/MOTUS-VIGILUS..git {REPO_DIR}

# Cacher le modèle MediaPipe sur Drive
MODEL_FILE = f"{MODELS_DIR}/pose_landmarker_heavy.task"
CODEBASE_MODEL = f"{REPO_DIR}/U-ALPHA/codebase/pose_landmarker_heavy.task"
if os.path.exists(MODEL_FILE):
    # Copier depuis Drive vers codebase (plus rapide que re-télécharger)
    import shutil
    shutil.copy2(MODEL_FILE, CODEBASE_MODEL)
    print(f"✅ Modèle MediaPipe chargé depuis Drive")
else:
    print("ℹ️ Le modèle MediaPipe sera téléchargé au premier run, puis caché sur Drive")

print(f"✅ Frégate U-ALPHA — Installation terminée")
print(f"📁 Inputs  : {INPUTS_DIR}")
print(f"📁 Outputs : {OUTPUTS_DIR}")

In [ ]:
# ═══════ CONFIGURATION ═══════
# @title Paramètres U-ALPHA
FPS_CIBLE = 30  # @param [30, 60, 120] {type:"raw"}
LISSAGE = "moyen"  # @param ["faible", "moyen", "brutal"] {type:"string"}
ROOT_MOTION = True  # @param {type:"boolean"}

print(f"⚙️ FPS: {FPS_CIBLE} | Lissage: {LISSAGE} | Root Motion: {ROOT_MOTION}")

In [ ]:
# ═══════ SÉLECTION VIDÉO ═══════
import glob

mp4_files = sorted(glob.glob(f"{INPUTS_DIR}/*.mp4") + glob.glob(f"{INPUTS_DIR}/*.MP4"))

if not mp4_files:
    print(f"❌ Aucune vidéo trouvée dans {INPUTS_DIR}")
    print("   Déposez vos fichiers .mp4 dans ce dossier Drive, puis relancez cette cellule")
else:
    print("📹 Vidéos disponibles :")
    for i, f in enumerate(mp4_files):
        print(f"  [{i}] {os.path.basename(f)}")
    
    # @title Sélectionner la vidéo
    VIDEO_INDEX = 0  # @param {type:"integer"}
    video_path = mp4_files[VIDEO_INDEX]
    print(f"\n✅ Vidéo sélectionnée : {os.path.basename(video_path)}")

In [ ]:
# ═══════ EXTRACTION — FRÉGATE U-ALPHA ═══════
import os

# Lancer l'extraction, sortie dans un dossier temporaire local (plus rapide)
LOCAL_OUT = "/content/alpha_outputs"
os.makedirs(LOCAL_OUT, exist_ok=True)

root_flag = "" if ROOT_MOTION else "--no-root-motion"
!python {REPO_DIR}/U-ALPHA/codebase/motus_extract.py "{video_path}" -o {LOCAL_OUT}/ --fps {FPS_CIBLE} --smooth {LISSAGE} {root_flag}

# Copier le modèle vers Drive pour le cacher (si c'est le 1er run)
import shutil
model_src = f"{REPO_DIR}/U-ALPHA/codebase/pose_landmarker_heavy.task"
if os.path.exists(model_src) and not os.path.exists(MODEL_FILE):
    shutil.copy2(model_src, MODEL_FILE)
    print(f"✅ Modèle MediaPipe caché sur Drive")

# Copier les outputs vers Drive
import glob
npz_files = sorted(glob.glob(f"{LOCAL_OUT}/motus_core_P*.npz"))
for f in npz_files:
    dest = f"{OUTPUTS_DIR}/{os.path.basename(f)}"
    shutil.copy2(f, dest)

print(f"\n✅ {len(npz_files)} fichier(s) .npz sauvegardés sur Drive :")
for f in npz_files:
    print(f"  📦 {OUTPUTS_DIR}/{os.path.basename(f)}")

In [ ]:
# ═══════ VÉRIFICATION ═══════
import glob

npz_on_drive = sorted(glob.glob(f"{OUTPUTS_DIR}/motus_core_P*.npz"))
print(f"📦 {len(npz_on_drive)} fichier(s) .npz sur Drive :")
for f in npz_on_drive:
    size_kb = os.path.getsize(f) / 1024
    print(f"  ✅ {os.path.basename(f)} ({size_kb:.1f} KB)")

print(f"\n⚔️ Frégate U-ALPHA — Mission accomplie.")
print(f"📋 Prochain pas : Copier les .npz dans Drive/MOTUS-VIGILUS/U-GAMMA/inputs/")
print(f"   puis ouvrir le notebook Frégate U-GAMMA")